# 04. 세 head × seed export
선택한 완료 checkpoint에서 backbone과 출처를 내보냅니다.

## 1. 설정
HEADS/SEEDS는 03과 동일해야 합니다.

In [ ]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
# 00의 공통값을 읽습니다. 아래 변수 변경이 저장값/YAML보다 우선합니다.
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])  # 예: [8972]. 이후 단계도 같은 목록을 사용하세요.
HEADS = list(saved["campaign"]["heads"])  # 기본 세 head. 02는 ArcFace 부모만 생성합니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]
NUM_WORKERS = 0
DEVICE = saved["project"]["device"]
DATASET = dict(saved["dataset"])  # 경로·input_size·pixel_mean/std
BACKBONE = dict(saved["backbone"])
TRAINING = dict(saved["training"])  # epoch·batch·학습률 등
HEAD_PARAMETERS = saved["heads"]
PATHS = dict(saved["paths"])
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]
EXECUTE = False  # True일 때만 목록 생성/학습/export를 수행합니다.
STAGE = "export"
# 현재 노트북 변수 -> 실제 모듈 설정. 입력 112x112 이외는 현 구조에서 명시적 중단.
DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
               test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
               meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
    "project": {"seed": SEEDS[0], "seed_list": SEEDS, "keep_raw_results": KEEP_RAW_RESULTS, "device": DEVICE},
    "campaign": {"heads": HEADS, "reuse_completed": REUSE_COMPLETED},
    "dataset": DATASET, "backbone": BACKBONE, "heads": HEAD_PARAMETERS,
    "training": {**TRAINING, "num_workers": NUM_WORKERS}, "paths": PATHS,
})


## 2. 대상

In [ ]:
plan = plan_matrix(settings,STAGE)
for row in plan:
    print(row)
print("TARGET_EPOCH:",TRAINING["epoches"],"예정:",len(plan))

## 3. 출처 기준

In [ ]:
print("전처리와 학습 조건은 source checkpoint 기록에서 가져옵니다.")
print("완료·seed·head·stage·설정 hash를 검증합니다.")

## 4. export

In [ ]:
from modules.notebook_workflow import export_matrix
results = export_matrix(settings) if EXECUTE else []
if not EXECUTE:
    print("계획 모드: export 미실행")

## 5. 결과

In [ ]:
for result in results:
    print(result)
print("완료 export:",sum(r.get("completed") is True for r in results),"/",len(plan))
print("strict load/shape 확인. 실제 ronbun·Continuous FIQA 평가는 미실행.")
print("해석 안내:",BASE_DIR/"fine-tune/README.md")